# Mamba from Scratch: Selective State Space Models

In this notebook, we'll implement **Mamba**, a state-space model that achieves transformer-like performance with linear complexity by using **selective** state space models.

## What You'll Learn

- How **state space models** (SSMs) work and why they're efficient
- The key innovation: **selection mechanism** (data-dependent parameters)
- How to implement the **selective scan** algorithm
- Building and training Mamba on text generation
- How Mamba compares to RWKV and transformers

## Key Intuition

Mamba asks: *"What if the model could dynamically decide what information to remember based on the input content?"* Traditional SSMs have fixed dynamics, but Mamba makes the state transition **data-dependent**, allowing it to selectively focus on relevant information and filter out noise.

## 1. Configuration

Set all hyperparameters in one place for easy experimentation.

**Note**: These settings are optimized for quick testing. Mamba's selective scan is computationally intensive, so we use smaller model dimensions and shorter sequences. For better results, try:
- `block_size`: 128 or 256
- `hidden_size`: 256 or 512
- `state_size`: 16 or 32
- `n_layers`: 2 or more
- `batch_size`: 64

In [ ]:
CONFIG = {
    # Data
    "block_size": 64,  # Sequence length (reduced for faster training)
    "vocab_size": None,  # Set after tokenization
    
    # Model architecture
    "n_embed": 128,  # Embedding dimension
    "hidden_size": 128,  # Hidden state dimension (d_model) - reduced for speed
    "state_size": 8,  # SSM state dimension (d_state) - reduced for speed
    "n_layers": 1,  # Number of Mamba layers (reduced for speed)
    "expand_factor": 2,  # Expansion factor for intermediate dimension
    
    # Training
    "batch_size": 32,  # Batch size (reduced for speed)
    "n_epochs": 20,  # Maximum training epochs (reduced for speed)
    "learning_rate": 1e-3,  # Optimizer learning rate
    "max_patience": 3,  # Early stopping patience
    "early_stop_threshold": 0.001,
}

## Detect Hardware

Use the aiml_notebooks library to detect available hardware and configure device.

In [ ]:
from aiml_notebooks.hardware import detect_hardware

hw = detect_hardware(verbose=True)
DEVICE = hw.device
print(f"\nUsing device: {DEVICE}")

## 2. Load and Prepare Data

We'll use the same Lord of the Rings text as in the RWKV notebook for direct comparison.

In [ ]:
with open("data/lotr.txt", "r", encoding="utf-8") as f:
    TEXT = f.read()
TEXT = TEXT.lower()
TEXT[:1000]

### Build Character-Level Vocabulary

Create mappings between characters and integers for tokenization.

In [ ]:
VOCAB = sorted(list(set(TEXT)))
CONFIG["vocab_size"] = len(VOCAB)
ctoi = dict((c, i) for i, c in enumerate(VOCAB))
itoc = dict((i, c) for i, c in enumerate(VOCAB))
encode = lambda str: [ctoi[c] for c in str]
decode = lambda tokens: "".join([itoc[i] for i in tokens])
len(VOCAB), decode(encode("hello world"))

### Tokenize the Text

Convert the entire text corpus into a sequence of integers.

In [ ]:
TOKENS = encode(TEXT)
print(f"Total tokens: {len(TOKENS)}")
TOKENS[:20]

## 3. Create Dataset

Split the token sequence into fixed-length chunks for training.

In [ ]:
import torch
from torch.utils.data import Dataset

class ChunkedDataset(Dataset):
    """Split token sequence into fixed-size chunks."""
    def __init__(self, tokens, block_size=CONFIG["block_size"]):
        tokens = torch.tensor(tokens)
        
        # Each chunk needs block_size+1 tokens (input + target)
        n_chunks = len(tokens) // (block_size + 1)
        tokens = tokens[:n_chunks * (block_size + 1)]
        
        self.chunks = tokens.view(n_chunks, block_size + 1)
    
    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        x = chunk[:-1]  # Input: all but last
        y = chunk[1:]   # Target: all but first (shifted by 1)
        return x, y
    
    def __len__(self):
        return len(self.chunks)

full_dataset = ChunkedDataset(TOKENS)
print(f"Total chunks: {len(full_dataset)}")
x_sample, y_sample = full_dataset[0]
print(f"Sample input shape: {x_sample.shape}")
print(f"Sample input: '{decode(x_sample.tolist())}'")
print(f"Sample target: '{decode(y_sample.tolist())}'")

### Split into Train and Validation Sets

Create a wrapper to subset the dataset without duplicating data.

In [ ]:
class DatasetSplit(Dataset):
    """Subset a dataset using indices."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = indices
    
    def __getitem__(self, idx):
        dataset_idx = self.indices[idx]
        return self.dataset[dataset_idx]
    
    def __len__(self):
        return len(self.indices)

### Create Train/Val Split

Randomly shuffle and split the dataset 80/20.

In [ ]:
import random
random.seed(42)

indices = list(range(len(full_dataset)))
random.shuffle(indices)
split_idx = int(len(indices) * 0.8)
train_indices = indices[:split_idx]
val_indices = indices[split_idx:]

train_dataset = DatasetSplit(full_dataset, train_indices)
val_dataset = DatasetSplit(full_dataset, val_indices)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")

## 4. Create DataLoaders

Batch the data for efficient training.

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    train_dataset,
    shuffle=True,
    batch_size=CONFIG["batch_size"]
)

val_dataloader = DataLoader(
    val_dataset,
    shuffle=False,
    batch_size=CONFIG["batch_size"]
)

# Verify shapes
x_batch, y_batch = next(iter(train_dataloader))
print(f"Batch input shape: {x_batch.shape}")  # (batch_size, block_size)
print(f"Batch target shape: {y_batch.shape}")  # (batch_size, block_size)

## 5. Understanding State Space Models and Mamba

### Classical State Space Models (SSMs)

A continuous-time SSM maps an input signal `u(t)` to an output `y(t)` through a hidden state `h(t)`:

```
h'(t) = A h(t) + B u(t)   (state evolution)
y(t)  = C h(t) + D u(t)   (output)
```

Where:
- **A**: State transition matrix (N×N)
- **B**: Input projection (N×1)
- **C**: Output projection (1×N)
- **D**: Skip connection (1×1)
- **h(t)**: Hidden state (N-dimensional)

### Discretization

To use SSMs in deep learning, we discretize them with step size Δ:

```
h_t = Ā h_{t-1} + B̄ x_t
y_t = C h_t
```

Where Ā and B̄ are discretized versions of A and B using the zero-order hold (ZOH) method.

### The Mamba Innovation: Selection

Traditional SSMs have **fixed** parameters A, B, C, Δ. Mamba makes them **input-dependent**:

```
B = Linear_B(x)  # Data-dependent input projection
C = Linear_C(x)  # Data-dependent output projection
Δ = Softplus(Linear_Δ(x))  # Data-dependent time step
```

This allows Mamba to:
1. **Filter irrelevant information**: Make B small for noise
2. **Reset state**: Make Δ large to forget the past
3. **Focus attention**: Make C large for important features

### Architecture Overview

```
Input x → Project to 2*d_inner
       ↓
Split into (x, z)
       ↓
x → Conv1d → Selective SSM → Output
                             ×
z → SiLU activation ─────────┘
       ↓
Project back to d_model
```

## 6. Implement SSM Discretization

Helper functions to discretize continuous SSM parameters using zero-order hold.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import math

def discretize_ssm(A, B, C, delta):
    """
    Discretize continuous SSM using zero-order hold (ZOH).
    
    Args:
        A: (d_state,) or (d_state, d_state) state matrix
        B: (batch, seq_len, d_state) input matrix
        C: (batch, seq_len, d_state) output matrix
        delta: (batch, seq_len) time step
    
    Returns:
        A_bar: Discretized state matrix
        B_bar: Discretized input matrix
    """
    # Ensure delta has the right shape for broadcasting
    # delta: (batch, seq_len, 1)
    delta = delta.unsqueeze(-1)
    
    # A is typically diagonal, shape (d_state,)
    # Discretize: Ā = exp(Δ * A)
    A_bar = torch.exp(delta * A)  # (batch, seq_len, d_state)
    
    # Discretize: B̄ = (Ā - I) / A * B
    # For numerical stability, use: B̄ = Δ * B (first-order approximation)
    B_bar = delta * B  # (batch, seq_len, d_state)
    
    return A_bar, B_bar

## 7. Implement Selective Scan

The core of Mamba: a recurrent scan with data-dependent parameters.

In [ ]:
def selective_scan(u, delta, A, B, C):
    """
    Perform selective scan (recurrent computation with data-dependent parameters).
    
    Args:
        u: (batch, seq_len, d_inner) input
        delta: (batch, seq_len, d_inner) time steps
        A: (d_inner, d_state) state matrix (log space)
        B: (batch, seq_len, d_state) input projection
        C: (batch, seq_len, d_state) output projection
    
    Returns:
        y: (batch, seq_len, d_inner) output
    """
    batch, seq_len, d_inner = u.shape
    d_state = A.shape[1]
    
    # Discretize for each channel separately
    # A: (d_inner, d_state) -> expand to (batch, seq_len, d_inner, d_state)
    A = A.unsqueeze(0).unsqueeze(0)  # (1, 1, d_inner, d_state)
    delta_A = delta.unsqueeze(-1) * A  # (batch, seq_len, d_inner, d_state)
    
    # Discretize A: exp(delta * A)
    A_bar = torch.exp(delta_A)  # (batch, seq_len, d_inner, d_state)
    
    # Discretize B: delta * B, then expand for d_inner
    # B: (batch, seq_len, d_state) -> (batch, seq_len, 1, d_state)
    # delta: (batch, seq_len, d_inner) -> (batch, seq_len, d_inner, 1)
    delta_B_u = delta.unsqueeze(-1) * B.unsqueeze(2) * u.unsqueeze(-1)
    # Result: (batch, seq_len, d_inner, d_state)
    
    # Initialize state
    h = torch.zeros(batch, d_inner, d_state, device=u.device, dtype=u.dtype)
    
    # Output tensor
    y = torch.zeros(batch, seq_len, d_inner, device=u.device, dtype=u.dtype)
    
    # Recurrent scan
    for t in range(seq_len):
        h = A_bar[:, t] * h + delta_B_u[:, t]  # (batch, d_inner, d_state)
        # C: (batch, seq_len, d_state) -> (batch, d_state)
        y[:, t] = (h * C[:, t].unsqueeze(1)).sum(dim=-1)  # (batch, d_inner)
    
    return y

## 8. Implement Mamba Block

The main building block combining selective SSM with gating and convolution.

In [ ]:
class MambaBlock(nn.Module):
    """Mamba block with selective SSM."""
    def __init__(self, d_model, d_state=16, expand=2, conv_kernel=4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state
        self.d_inner = d_model * expand
        self.conv_kernel = conv_kernel
        
        # Input projection (project to 2 * d_inner for x and z)
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        
        # Convolution (adds local context)
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=conv_kernel,
            padding=conv_kernel - 1,
            groups=self.d_inner  # Depthwise convolution
        )
        
        # SSM parameters
        # A is initialized as log of complex numbers (S4D-Lin initialization)
        A = torch.arange(1, d_state + 1, dtype=torch.float32).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))  # Keep in log space for stability
        
        # Learnable parameters for selective mechanism
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + 1, bias=False)  # For B, C, delta
        
        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, d_model)
        Returns:
            output: (batch, seq_len, d_model)
        """
        batch, seq_len, d_model = x.shape
        
        # Input projection and split
        x_and_z = self.in_proj(x)  # (batch, seq_len, 2 * d_inner)
        x, z = x_and_z.chunk(2, dim=-1)  # Each: (batch, seq_len, d_inner)
        
        # Convolution (needs channels-first format)
        x_conv = self.conv1d(x.transpose(1, 2))[:, :, :seq_len].transpose(1, 2)
        # Result: (batch, seq_len, d_inner)
        
        # Activation
        x_conv = F.silu(x_conv)
        
        # Generate selective parameters (B, C, delta) from input
        x_proj = self.x_proj(x_conv)  # (batch, seq_len, d_state * 2 + 1)
        delta, B, C = torch.split(
            x_proj,
            [1, self.d_state, self.d_state],
            dim=-1
        )
        # delta: (batch, seq_len, 1)
        # B, C: (batch, seq_len, d_state)
        
        # Ensure delta is positive and expand to d_inner
        delta = F.softplus(delta).squeeze(-1).unsqueeze(-1).expand(-1, -1, self.d_inner)
        # delta: (batch, seq_len, d_inner)
        
        # Selective scan with data-dependent parameters
        A = -torch.exp(self.A_log)  # (d_inner, d_state) - negative for stability
        y = selective_scan(x_conv, delta, A, B, C)
        # y: (batch, seq_len, d_inner)
        
        # Gating mechanism
        y = y * F.silu(z)
        
        # Output projection
        output = self.out_proj(y)
        return output

### Test Mamba Block

Verify the Mamba block processes sequences correctly.

In [ ]:
# Test with random input
test_input = torch.randn(2, 10, CONFIG["hidden_size"])  # (B=2, T=10, d_model)
mamba_block = MambaBlock(
    d_model=CONFIG["hidden_size"],
    d_state=CONFIG["state_size"],
    expand=CONFIG["expand_factor"]
)
test_output = mamba_block(test_input)
print(f"Input shape: {test_input.shape}")
print(f"Output shape: {test_output.shape}")
print(f"Output preserves shape: {test_output.shape == test_input.shape}")

## 9. Implement Residual Mamba Block

Wrap Mamba block with layer normalization and residual connection.

In [ ]:
class ResidualMambaBlock(nn.Module):
    """Mamba block with pre-norm and residual connection."""
    def __init__(self, d_model, d_state=16, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MambaBlock(d_model, d_state, expand)
    
    def forward(self, x):
        # Pre-norm with residual connection
        return x + self.mamba(self.norm(x))

## 10. Implement Full Mamba Model

Stack multiple Mamba blocks with embedding and output layers.

In [ ]:
class Mamba(nn.Module):
    """Full Mamba model for language modeling."""
    def __init__(
        self,
        vocab_size=CONFIG["vocab_size"],
        n_embed=CONFIG["n_embed"],
        d_model=CONFIG["hidden_size"],
        d_state=CONFIG["state_size"],
        n_layers=CONFIG["n_layers"],
        expand=CONFIG["expand_factor"]
    ):
        super().__init__()
        
        # Embedding layer
        self.embeddings = nn.Embedding(vocab_size, n_embed)
        
        # Project embeddings to model dimension if different
        self.embed_proj = nn.Linear(n_embed, d_model, bias=False) if n_embed != d_model else nn.Identity()
        
        # Stack of Mamba blocks
        self.blocks = nn.ModuleList([
            ResidualMambaBlock(d_model, d_state, expand)
            for _ in range(n_layers)
        ])
        
        # Output layer
        self.ln_out = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
    
    def forward(self, x):
        # Embed tokens
        x = self.embeddings(x)  # (B, T) -> (B, T, n_embed)
        x = self.embed_proj(x)  # (B, T, n_embed) -> (B, T, d_model)
        
        # Pass through Mamba blocks
        for block in self.blocks:
            x = block(x)  # (B, T, d_model) -> (B, T, d_model)
        
        # Output projection
        x = self.ln_out(x)  # (B, T, d_model)
        logits = self.head(x)  # (B, T, d_model) -> (B, T, vocab_size)
        
        return logits

### Initialize and Test Model

Create the model and verify it processes a batch correctly.

In [ ]:
model = Mamba().to(DEVICE)

# Test forward pass
x_test, _ = next(iter(train_dataloader))
x_test = x_test.to(DEVICE)
logits = model(x_test)

print(f"Input shape: {x_test.shape}")
print(f"Output logits shape: {logits.shape}")
print(f"Expected shape: (batch={CONFIG['batch_size']}, seq_len={CONFIG['block_size']}, vocab={CONFIG['vocab_size']})")

# Count parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {n_params:,}")

## 11. Implement Evaluation Function

Compute validation loss without updating gradients.

In [ ]:
from tqdm import tqdm

@torch.no_grad()
def evaluate(model, loader):
    """Evaluate model on a dataset."""
    model.eval()
    losses = []
    
    device = next(model.parameters()).device
    for x, y in tqdm(loader, disable=True):
        x, y = x.to(device), y.to(device)
        logits = model(x)  # (B, T, V)
        logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
        y = y.reshape(-1)  # (B*T,)
        loss = F.cross_entropy(logits, y)
        losses.append(loss.item())
    
    return sum(losses) / len(losses)

# Test evaluation
initial_loss = evaluate(model, val_dataloader)
print(f"Initial validation loss: {initial_loss:.4f}")
print(f"Random chance loss (log({CONFIG['vocab_size']})): {torch.log(torch.tensor(CONFIG['vocab_size'])).item():.4f}")

## 12. Training Loop

Train the Mamba model with early stopping based on validation loss.

In [ ]:
import time
import torch.optim

start = time.time()
n_epochs = CONFIG["n_epochs"]
lr = CONFIG["learning_rate"]
early_stop_threshold = CONFIG['early_stop_threshold']
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

train_losses = []
val_losses = []
patience = max_patience = CONFIG["max_patience"]

device = next(model.parameters()).device
best_val_loss = 999999999
for epoch in range(n_epochs):
    model.train()
    losses = []
    pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{n_epochs}")
    
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        
        # Forward pass
        logits = model(x)
        logits = logits.reshape(-1, logits.size(-1))  # (B*T, V)
        y = y.reshape(-1)  # (B*T,)
        loss = F.cross_entropy(logits, y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix({"train_loss": loss.item()})
    
    # Compute epoch metrics
    train_loss = sum(losses) / len(losses)
    val_loss = evaluate(model, val_dataloader)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}, patience={patience}")
    
    if len(val_losses) > 1:
        # Calculate improvement BEFORE updating best_val_loss
        val_loss_delta = best_val_loss - val_loss
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            # Only reset patience if improvement is significant
            if val_loss_delta > early_stop_threshold:
                patience = max_patience
            # If improvement is small (< threshold), still decrease patience
            else:
                patience -= 1
                print(f"Patience = {patience}, val_loss_delta = {val_loss_delta:.4f} (small improvement)")
        else:
            # No improvement - decrease patience
            patience -= 1
            print(f"Patience = {patience}, val_loss_delta = {val_loss_delta:.4f} (no improvement)")
        
        if patience == 0: 
            print("Early stopping due to val_loss stall")
            break

elapsed = time.time() - start
print(f"\nTraining finished in {elapsed:.1f}s")
print(f"Final train loss: {train_losses[-1]:.4f}")
print(f"Final val loss: {val_losses[-1]:.4f}")

## 13. Plot Training Curves

Visualize how the model learned over time.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Train Loss', marker='o', linewidth=2)
plt.plot(val_losses, label='Validation Loss', marker='s', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Mamba Training Progress', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Best validation loss: {min(val_losses):.4f} at epoch {val_losses.index(min(val_losses)) + 1}")

## 14. Text Generation

Test the trained model by generating text from a prompt.

In [ ]:
def generate(model, prompt, max_len=100, temperature=1.0):
    """Generate text continuation from a prompt."""
    model.eval()

    tokens = encode(prompt)
    max_new_tokens = max_len - len(tokens)
    assert max_new_tokens > 0, "max_len must be greater than prompt length"
    
    print(prompt, end="")
    
    device = next(model.parameters()).device
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Get predictions for current sequence
            tokens_t = torch.tensor(tokens).unsqueeze(0).to(device)  # (1, T)
            logits = model(tokens_t)  # (1, T, vocab_size)
            logits = logits[:, -1, :] / temperature  # (1, vocab_size) - only last position
            
            # Sample next token
            probs = F.softmax(logits, dim=-1)
            token = torch.multinomial(probs, num_samples=1).squeeze().item()
            
            # Append and print
            tokens.append(token)
            print(decode([token]), end="")
    
    print()  # Newline at end

### Generate Sample Text

Try generating text from different prompts.

In [ ]:
# Generate from a prompt
generate(model, "frodo picked up the ", max_len=100)

### Try Another Prompt

Generate text with different starting text.

In [ ]:
# Try another prompt
generate(model, "gandalf said ", max_len=100)

### Try with Lower Temperature

Use a lower temperature to generate more focused text.

In [ ]:
# Try with lower temperature (more focused)
generate(model, "the ring of power ", max_len=100, temperature=0.8)

## 15. Key Takeaways

### What We Learned

1. **State Space Models (SSMs)** provide an efficient way to model sequences with linear complexity O(T)

2. **Selection is the key innovation**: By making SSM parameters (B, C, Δ) input-dependent, Mamba can:
   - Filter irrelevant information
   - Reset state when needed
   - Focus on important features

3. **Selective scan** replaces attention: Instead of computing attention over all positions, Mamba uses a recurrent state with data-dependent dynamics

4. **Hardware-efficient**: The selective scan can be computed efficiently using parallel scan algorithms on GPUs

5. **Gating and convolution**: Like modern architectures, Mamba uses:
   - Gated linear units (GLU) for better expressiveness
   - Depthwise convolution for local context

### Mamba vs RWKV vs Transformers

| Aspect | Transformer | RWKV | Mamba |
|--------|------------|------|-------|
| Complexity | O(T²) | O(T) | O(T) |
| Mechanism | Attention | Linear Attention | Selective SSM |
| Parameters | Fixed | Mixed (time decay) | Data-dependent (B, C, Δ) |
| Memory | High | Low | Low |
| Parallelizable | Yes | Yes (training) | Yes (training) |

### Why This Matters

Mamba demonstrates that:
- **Selection** (data-dependent parameters) is crucial for sequence modeling
- SSMs can be competitive with attention mechanisms
- Linear complexity doesn't require sacrificing expressiveness

### Further Exploration

- Compare performance with RWKV on the same task
- Visualize the learned Δ, B, C parameters to understand selection
- Implement the hardware-efficient parallel scan
- Try different SSM initialization schemes (S4D, S4D-Lin, etc.)
- Scale up to larger models and longer sequences